In [3]:
import pandas as pd
import re

def clean_data():
    df = pd.read_csv('./raw_books2.csv')
    print(f"Loaded {len(df)} raw records.")

    df['title'] = df['title'].astype(str).str.strip()
    df['category'] = df['category'].astype(str).str.strip()
    df['description'] = df['description'].astype(str).str.strip()
    df['upc'] = df['upc'].astype(str).str.strip()

    df = df.drop_duplicates(subset=['upc']).reset_index(drop=True)

    df['price'] = df['price'].astype(str).str.extract(r'(\d+\.\d+)')[0].astype(float)

    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    df['rating'] = df['rating'].astype(str).str.strip().map(rating_map)

    def extract_stock(val):
        match = re.search(r'\((\d+)\s*available\)', str(val))
        if match:
            return int(match.group(1))
        elif 'In stock' in str(val):
            return 1
        return 0

    df['stock_count'] = df['availability'].apply(extract_stock)

    # 7. Feature engineering
    df['description_word_count'] = df['description'].apply(lambda x: len(str(x).split()))
    df['price_band'] = pd.qcut(df['price'], q=3, labels=['Budget', 'Moderate', 'Expensive'])
    df['value_score'] = (df['rating'] / df['price'] * 100).round(2)

    clean_cols = [
        'title', 'category', 'price', 'rating', 'stock_count',
        'description', 'upc', 'number_of_reviews', 'product_url',
        'description_word_count', 'price_band', 'value_score'
    ]
    
    df_clean = df[clean_cols]
    df_clean.to_csv('cleaned_books.csv', index=False)
    
    print("Preprocessing complete! Exported to 'cleaned_books.csv'.")
    display(df_clean[['title', 'price', 'rating', 'stock_count', 'price_band', 'value_score']].head())

clean_data()

Loaded 100 raw records.
Preprocessing complete! Exported to 'cleaned_books.csv'.


,title,price,rating,stock_count,price_band,value_score
0,It's Only the Himalayas,45.17,2,19,Expensive,4.43
1,Libertarianism for Beginners,51.33,2,19,Expensive,3.90
2,Mesaerion: The Best Science Fiction Stories 18...,37.59,1,19,Moderate,2.66
3,Olio,23.88,1,19,Moderate,4.19
4,Our Band Could Be Your Life: Scenes from the A...,57.25,3,19,Expensive,5.24
